# Arranque en Colab

Deja listo el entorno de referencia: una GPU T4, Ollama 0.40.10, el modelo de `config.toml` y los datos del Mundial. Corré las celdas **en orden**. Tarda unos 10 minutos la primera vez, casi todo en bajar el modelo.

**Antes de empezar:** elegí la GPU T4 en *Entorno de ejecución → Cambiar tipo de entorno de ejecución*. Cambiarla después reemplaza la máquina y borra lo que hicieron las celdas siguientes. Desde VS Code: *Select Kernel → Colab → New Colab Server* y elegí la T4.

El disco de Colab se borra al desconectarse. Subí `trazas/` y `resultados/` a GitHub antes de cerrar (sección 6).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader || echo "sin GPU: cambiá el entorno a T4" 

## 1 · Tu código

Tu repositorio es privado: Colab necesita un **token** de GitHub para bajarlo y para subir tus trazas. Crealo así:

1. GitHub → tu foto → *Settings* → *Developer settings* → *Personal access tokens* → *Fine-grained tokens* → *Generate new token*.
2. *Repository access*: *Only select repositories*, y elegí tu repositorio.
3. *Repository permissions* → **Contents: Read and write**. Poné un vencimiento corto.
4. Copiá el token: GitHub lo muestra una sola vez.

Poné tu usuario y el nombre de tu repositorio en `REPO`. La celda te pide el token y **no lo guarda en el cuaderno**; no lo pegues en ningún archivo.

In [ ]:
import os
from getpass import getpass

REPO = "TU_USUARIO/TU_REPO"   # <- cambiá esto
CARPETA = "/content/proyecto"

assert "TU_USUARIO" not in REPO, "Poné tu usuario y el nombre de tu repositorio en REPO"
os.chdir("/content")
if not os.path.exists(CARPETA):
    # El token queda en la configuración de git de esta máquina, que Colab borra al desconectarse.
    REPO_URL = f"https://{getpass('Token de GitHub: ')}@github.com/{REPO}.git"
    !git clone $REPO_URL $CARPETA
    del REPO_URL
os.chdir(CARPETA)
!git pull --rebase
!git log --oneline -1

## 2 · Ollama, con la versión fijada
El instalador baja la versión que le pidas con `OLLAMA_VERSION`. Sin eso baja la última, y los tiempos y el formato de las llamadas a herramientas pueden cambiar.

El instalador avisa «Unable to detect NVIDIA/AMD GPU» y `apt` un aviso sobre `r2u.stat.illinois.edu`: son normales. La GPU se chequea de verdad en la sección 4.

In [ ]:
OLLAMA_VERSION = "0.40.10"
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=$OLLAMA_VERSION sh

In [ ]:
import os, subprocess, time, requests

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"   # un solo modelo en la placa
os.environ["OLLAMA_NUM_PARALLEL"] = "1"        # un pedido a la vez: tiempos comparables

def ollama_vivo():
    try:
        return requests.get("http://127.0.0.1:11434/api/tags", timeout=2).ok
    except requests.exceptions.RequestException:
        return False

if not ollama_vivo():
    # start_new_session: interrumpir una celda no le manda SIGINT al servidor.
    subprocess.Popen(["ollama", "serve"], start_new_session=True,
                     stdout=open("/content/ollama.log", "a"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_vivo():
            break
        time.sleep(1)
print("Ollama está listo." if ollama_vivo() else "El servidor no levantó: mirá /content/ollama.log")

## 3 · El modelo y los datos

In [ ]:
import config

MODELO = config.cargar()["modelo"]["nombre"]
!ollama pull $MODELO
!python datos/descargar.py

## 4 · Chequeos
Tres cosas que conviene mirar antes de medir nada: la versión de Ollama, que el modelo corra **100% en la GPU**, y que la configuración sea válida.

La primera llamada tarda más de un minuto: carga el modelo (6 GB) en la GPU. Las siguientes son rápidas.

In [ ]:
import llm

cfg = config.cargar()
print("Ollama", llm.version(), "(esperada: " + OLLAMA_VERSION + ")")
if llm.version() != OLLAMA_VERSION:
    print("   ATENCIÓN: no es la versión del kit; decilo en tu informe.")

# Una llamada de prueba para que el modelo quede cargado, y después dónde corre.
respuesta = llm.chat([{"role": "user", "content": "Respondé solo con la palabra: listo"}],
                     modelo=MODELO, num_ctx=cfg["modelo"]["num_ctx"], num_predict=20)
print("respuesta:", respuesta.mensaje["content"].strip(), "·", respuesta.tokens_entrada, "tokens leídos,",
      respuesta.tokens_salida, "escritos,", respuesta.duracion_ms, "ms")
ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout
print(ps)
if "100% GPU" not in ps:
    print("ATENCIÓN: el modelo no corre 100% en la GPU. Eso cambia los tiempos: revisá /content/ollama.log.")

In [ ]:
!pip install -q pytest
!python -m pytest -q

## 5 · Corré el kit

Con el esqueleto sin implementar, `agente.py` termina con estado `error` y `evaluar.py` da 0 aciertos. Es el punto de partida: el runner imprime `fallo` en las 30 tareas y eso es lo esperado hasta que escribas el loop. La línea de base a libro cerrado sí funciona desde ahora.

En Colab los comandos se escriben con `python`, no con `uv run`, aunque el script imprima esa sugerencia.

```
!python libro_cerrado.py --corrida trazas/libro_cerrado
!python evaluar.py --corrida trazas/libro_cerrado

!python correr_todas.py --corrida trazas/mi_corrida
!python evaluar.py --corrida trazas/mi_corrida
```

## 6 · Guardar tus trazas en GitHub

El disco de Colab se borra al desconectarse. Para conservar una corrida, subila a tu repositorio. Copiá esto en una celda nueva y poné tu nombre y tu mail (git los pide para firmar el commit):

```
%cd /content/proyecto
!git config user.email "TU_MAIL"
!git config user.name "TU_NOMBRE"
!git add trazas resultados
!git commit -m "trazas de la corrida"
!git pull --rebase
!git push
```

El `git pull --rebase` antes del `git push` es necesario si subiste cambios desde tu máquina después de clonar: sin él, GitHub rechaza el `push`. Después, en tu máquina, `git pull` trae las trazas.